# Thinking-mode diagnostic — no training

Can thinking mode help the same model on arithmetic and the remaining development errors? Evaluate ten arithmetic questions on the base and adapter, plus all 84 F/R development questions on the adapter. Keep a greedy reproduction arm, then compare sampled non-thinking and sampled thinking with identical sampling settings and a 1,024-token cap: **312 answers, zero training updates**.

This notebook saves generated visible reasoning text and scores the final answer separately. An unfinished thinking block cannot count as a correct answer. Visible text is not proof of internal reasoning, and M1 traces are not post-unlearning leakage results. Original pilot gates/results remain unchanged.

Use the existing Kaggle GPU runtime with HF_TOKEN attached. **Publish the new code, config and verified utility-run reference files to GitHub main before executing this notebook.**


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='thinking-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/mode_diagnostic.py').is_file(), 'Publish the new thinking-mode diagnostic files before running this notebook.'
assert Path('experiments/runs/m1_utility_prompt_diagnostic_20261105/base_arithmetic-v1.jsonl').is_file(), 'Publish the verified utility reference records too.'


In [ ]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


In [ ]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='thinking-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.mode_diagnostic', '--config', 'configs/evaluation/thinking_mode_diagnostic_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['prediction_count'] == 312 and plan['training_updates'] == 0
print('Reference hashes passed. Planned answers:', plan['prediction_count'])


In [ ]:
from kaggle_secrets import UserSecretsClient

evaluation_env = os.environ.copy()
evaluation_env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
output_dir = Path(tempfile.mkdtemp(prefix='m1-thinking-mode-', dir=remote_root))
print('Evaluation outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.mode_diagnostic', '--config', 'configs/evaluation/thinking_mode_diagnostic_v1.json', '--output-dir', str(output_dir), '--mode', 'evaluate'], env=evaluation_env, check=True)
finally:
    evaluation_env.pop('HF_TOKEN', None)


In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
print('Greedy controls reproduced:', summary['greedy_reproduction'])
for name, group in sorted(summary['groups'].items()):
    print(name, '| strict:', str(group['answer_matches']) + '/' + str(group['count']), '| caps:', group['token_limit_hits'], '| missing thinking boundaries:', group['missing_thinking_boundaries'])
    if 'learning_metrics' in group:
        print('Development groups:', json.dumps(group['learning_metrics']))
for checkpoint in ['base', 'adapter']:
    file = output_dir / (checkpoint + '_utility_sampled_thinking.jsonl')
    rows = [json.loads(line) for line in file.read_text().splitlines()]
    print('\n', checkpoint, 'example visible reasoning:', rows[0]['visible_reasoning'][:1500])
    print('Example final answer:', rows[0]['final_answer_text'])
    for row in rows:
        print(row['example_id'], '| expected:', row['answer'], '| final:', repr(row['final_answer_text']), '| correct:', row['answer_match'])
print('One sampling seed and bounded outputs: inspect caps/boundaries and raw evidence before drawing conclusions. Original pilot remains failed.')


In [ ]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
